# Task 1: Counted PCFG and CKY

From CKY to Neural Grammar Induction · Penn Treebank, Partition A

## Settings

Run the cells in order. Task 1 estimates its grammar from the training trees.

In [1]:
from pathlib import Path

seed = 42
train_compound_model = False
train_supervised_model = False
download_models = False
model_dir = Path("checkpoints")
results_dir = Path("results")

In [2]:
import hashlib
import importlib.util
import json
import random
import subprocess
import sys
import zipfile
from collections import Counter, defaultdict
from datetime import datetime
from importlib.metadata import version
from math import log

packages = {
    "nltk": "nltk==3.9.1",
    "torch_struct": "torch-struct==0.5",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
}
if download_models:
    packages["gdown"] = "gdown==5.2.0"

missing = [package for module, package in packages.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])

import matplotlib.pyplot as plt
import nltk
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from nltk.corpus import treebank
from nltk.tree import Tree
from torch import nn
from torch.nn import functional as F
from torch_struct import SentCFG

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

run_dir = results_dir / datetime.now().strftime("%Y%m%d_%H%M%S_%f")
run_dir.mkdir(parents=True)
new_model_dir = run_dir / "trained_checkpoints"
new_model_dir.mkdir()

In [3]:
saved_models = {
    "task2": (
        "task2_best_epoch1_saved.pt",
        "1yDXrNZwlQzL2ifp_vFX9xEn0WlC8G4vL",
        "237073d7409caf1e9093b7db816f6686a9f87c15bf1b9b17c975b39b0fc1c87e",
    ),
    "task3": (
        "task3_epoch15.pt",
        "1TSyhTKk_6pZg8I6Lz2jJtEObzL6RwpCU",
        "73929d6170a1c937798fd628629e495dd11acc4e54f7ed1d709ecf0690d07352",
    ),
}


def file_hash(path):
    with path.open("rb") as file:
        return hashlib.file_digest(file, "sha256").hexdigest()


def checkpoint_path(task):
    filename, drive_id, expected_hash = saved_models[task]
    path = model_dir / filename

    if not path.exists():
        if not download_models:
            raise FileNotFoundError(f"Expected checkpoint at {path}")
        import gdown

        model_dir.mkdir(parents=True, exist_ok=True)
        temporary_file = path.with_suffix(".part")
        try:
            gdown.download(id=drive_id, output=str(temporary_file), use_cookies=False)
            if not temporary_file.exists() or file_hash(temporary_file) != expected_hash:
                raise ValueError(f"The download for {filename} did not match the saved model.")
            temporary_file.replace(path)
        finally:
            temporary_file.unlink(missing_ok=True)

    if download_models and file_hash(path) != expected_hash:
        raise ValueError(f"{path} differs from the shared checkpoint.")
    return path


def save_json(filename, value):
    (run_dir / filename).write_text(json.dumps(value, indent=2, ensure_ascii=False),
                                   encoding="utf-8")


package_names = ["numpy", "nltk", "torch", "torch-struct", "pandas", "matplotlib"]
if download_models:
    package_names.append("gdown")
package_versions = {package: version(package) for package in package_names}
run_info = {
    "seed": seed,
    "device": str(device),
    "python": sys.version.split()[0],
    "packages": package_versions,
    "checkpoints": {},
    "task2_mode": "train" if train_compound_model else "checkpoint",
    "task3_mode": "train" if train_supervised_model else "checkpoint",
    "evaluation": "Unique unlabeled spans after punctuation projection; no full-sentence or single-word spans.",
}


def record_checkpoint(task, path, checkpoint):
    run_info["checkpoints"][task] = {
        "filename": path.name,
        "epoch": checkpoint["epoch"],
        "sha256": file_hash(path),
        "validation_loss": checkpoint.get("best_validation_loss"),
    }

## Data and split

I shuffle the 3,914 trees with seed 42, take the first 1,305 for Partition A, and shuffle that partition again before splitting it. I remove empty categories and strip functional suffixes such as NP-SBJ. The words, case and punctuation stay in the model input.

In [4]:
def clean_tree(node):
    if isinstance(node, str):
        return node
    if node.label() == "-NONE-":
        return None

    children = [clean_tree(child) for child in node]
    children = [child for child in children if child is not None]
    if not children:
        return None

    label = node.label()
    if not label.startswith("-"):
        label = label.split("-", 1)[0].split("=", 1)[0]
    return Tree(label, children)


try:
    all_trees = list(treebank.parsed_sents())
except LookupError:
    nltk.download("treebank", quiet=True)
    all_trees = list(treebank.parsed_sents())

assert len(all_trees) == 3914
indices = np.random.RandomState(seed).permutation(len(all_trees))
partitions = {
    "A": indices[:1305].tolist(),
    "B": indices[1305:2610].tolist(),
    "C": indices[2610:].tolist(),
}
partition_a = np.random.RandomState(seed).permutation(partitions["A"])
train_end = int(0.8 * len(partition_a))
validation_end = train_end + int(0.1 * len(partition_a))
split_indices = {
    "train": partition_a[:train_end].tolist(),
    "validation": partition_a[train_end:validation_end].tolist(),
    "test": partition_a[validation_end:].tolist(),
}
trees_by_split = {
    split: [clean_tree(all_trees[index]) for index in tree_indices]
    for split, tree_indices in split_indices.items()
}
train_trees = trees_by_split["train"]
validation_trees = trees_by_split["validation"]
test_trees = trees_by_split["test"]

save_json("partition_indices.json", {"partitions": partitions, "partition_a_splits": split_indices})
split_text = json.dumps(split_indices, sort_keys=True, separators=(",", ":"))
run_info["split_sha256"] = hashlib.sha256(split_text.encode()).hexdigest()
display(pd.DataFrame([
    {"Split": split.title(), "Trees": len(trees)} for split, trees in trees_by_split.items()
]))

,Split,Trees
0,Train,1044
1,Validation,130
2,Test,131


In [5]:
def replace_rare_words(trees, cutoff):
    word_counts = Counter(word for tree in trees for word in tree.leaves())
    changed_trees = []
    for tree in trees:
        changed = tree.copy(deep=True)
        for position in changed.treepositions("leaves"):
            if word_counts[changed[position]] <= cutoff:
                changed[position] = "<UNK>"
        changed_trees.append(changed)
    return changed_trees


def to_cnf(tree, markov_order):
    binary_tree = tree.copy(deep=True)
    binary_tree.collapse_unary(collapsePOS=True, collapseRoot=True)
    binary_tree.chomsky_normal_form(factor="right", horzMarkov=markov_order)
    return binary_tree

## How I score the parses

A bracket records where a phrase starts and ends. Labeled F1 also checks its category, such as NP. The original Task 1 scores keep punctuation. For the comparison across all three tasks, I remove punctuation using the gold POS tags, count each remaining span once, and exclude single words and the full sentence. A sentence with fewer than two content words is skipped. A failed parse on an eligible sentence still counts against recall.

In [6]:
punctuation_tags = {".", ",", ":", "``", "''", "-LRB-", "-RRB-"}


def brackets(tree, labeled=False):
    spans = Counter()

    def visit(node, start, is_root=False):
        if isinstance(node, str):
            return start + 1
        end = start
        for child in node:
            end = visit(child, end)
        if not is_root and any(isinstance(child, Tree) for child in node):
            span = (start, end, node.label()) if labeled else (start, end)
            spans[span] += 1
        return end

    if tree is not None:
        visit(tree, 0, is_root=True)
    return spans


def content_length(tree):
    return sum(tag not in punctuation_tags for _, tag in tree.pos())


def project_spans(spans, prefix):
    # prefix[i] counts the content words before original position i.
    projected = set()
    for start, end in spans:
        left, right = prefix[start], prefix[end]
        if 2 <= right - left < prefix[-1]:
            projected.add((left, right))
    return projected


def score_sentence(split, index, gold_tree, prediction, predicted_spans=None, one_sided=None):
    words = gold_tree.leaves()
    gold_tags = gold_tree.pos()
    prefix = [0]
    for _, tag in gold_tags:
        prefix.append(prefix[-1] + int(tag not in punctuation_tags))

    eligible = prefix[-1] >= 2
    if prediction is not None:
        assert prediction.leaves() == words

    gold = project_spans(brackets(gold_tree), prefix)
    predicted = project_spans(
        brackets(prediction) if predicted_spans is None else predicted_spans, prefix
    )
    matches = gold & predicted
    span_count = len(gold) + len(predicted)
    sentence_f1 = 2 * len(matches) / span_count if span_count else 1.0

    return {
        "position": index, "corpus_index": split_indices[split][index],
        "tokens": words,
        "scored_words": [word for word, tag in gold_tags if tag not in punctuation_tags],
        "eligible": eligible, "complete": prediction is not None, "one_sided": one_sided,
        "matching": len(matches), "predicted_count": len(predicted), "gold_count": len(gold),
        "sentence_f1": sentence_f1 if eligible else None,
        "gold_spans": sorted(gold), "predicted_spans": sorted(predicted),
        "matching_spans": sorted(matches),
        "extra_spans": sorted(predicted - gold), "missed_spans": sorted(gold - predicted),
        "gold_tree": gold_tree.pformat(margin=10**9),
        "predicted_tree": prediction.pformat(margin=10**9) if prediction is not None else None,
    }


def bracket_scores(matches, predicted, gold):
    precision = matches / predicted if predicted else 0.0
    recall = matches / gold if gold else 0.0
    f1 = 2 * matches / (predicted + gold) if predicted + gold else 0.0
    return precision, recall, f1


def save_evaluation(name, records):
    scored = [row for row in records if row["eligible"]]
    matches = sum(row["matching"] for row in scored)
    predicted = sum(row["predicted_count"] for row in scored)
    gold = sum(row["gold_count"] for row in scored)
    precision, recall, corpus_f1 = bracket_scores(matches, predicted, gold)
    diagnosed = [row["one_sided"] for row in scored if row["one_sided"] is not None]
    summary = {
        "sentences": len(records), "scored": len(scored),
        "skipped": len(records) - len(scored),
        "complete": sum(row["complete"] for row in scored),
        "matching_spans": matches, "predicted_spans": predicted, "gold_spans": gold,
        "precision": precision, "recall": recall, "corpus_f1": corpus_f1,
        "sentence_f1": sum(row["sentence_f1"] for row in scored) / len(scored),
        "one_sided": sum(diagnosed), "long_sentences_diagnosed": len(diagnosed),
    }
    save_json(f"{name}_metrics.json", summary)
    with (run_dir / f"{name}_predictions.jsonl").open("w", encoding="utf-8") as file:
        for row in records:
            file.write(json.dumps(row, ensure_ascii=False) + "\n")
    return summary


def score_row(name, result):
    return {
        "Model": name,
        "Scored": result["scored"],
        "Complete parses": result["complete"],
        "Precision (%)": 100 * result["precision"],
        "Recall (%)": 100 * result["recall"],
        "Sentence F1 (%)": 100 * result["sentence_f1"],
        "Corpus F1 (%)": 100 * result["corpus_f1"],
    }


def show_examples(records, task):
    examples = [row for row in records if row["eligible"] and row["gold_count"]
                and 5 <= len(row["tokens"]) <= 25]
    if len(examples) < 2:
        return
    better = max(examples, key=lambda row: row["sentence_f1"])
    worse = min((row for row in examples if row is not better),
                key=lambda row: row["sentence_f1"])

    for name, row in [("High F1", better), ("Low F1", worse)]:
        print(f"{name} · test sentence {row['position']} · F1 {row['sentence_f1']:.1%}")
        print(" ".join(row["tokens"]))
        print("Gold:     ", Tree.fromstring(row["gold_tree"]).pformat(margin=95))
        predicted = row["predicted_tree"]
        print("Predicted:", Tree.fromstring(predicted).pformat(margin=95) if predicted else "No parse")
        words = row["scored_words"]
        for label, key in [("Matched", "matching_spans"), ("Missed", "missed_spans"),
                           ("Extra", "extra_spans")]:
            phrases = [" ".join(words[i:j]) for i, j in row[key]]
            print(f"{label}: {', '.join(phrases) or '—'}")
        print()

    save_json(f"{task}_examples.json", [better, worse])

## Task 1: Counted PCFG and CKY

I count rules after collapsing unary chains and binarizing the trees. Counts below 5 get one extra count before I normalize by the parent. I try unknown word cutoffs 1 and 2 and select the cutoff for each Markov order using labeled validation F1.

In [7]:
def fit_counted_pcfg(cnf_trees):
    rule_counts = Counter(rule for tree in cnf_trees for rule in tree.productions())
    smoothed = {rule: count + int(count < 5) for rule, count in rule_counts.items()}
    parent_totals = Counter()
    for rule, count in smoothed.items():
        parent_totals[rule.lhs()] += count

    lexical_rules = defaultdict(list)
    binary_rules = defaultdict(lambda: defaultdict(list))
    for rule, count in smoothed.items():
        parent = rule.lhs().symbol()
        log_prob = log(count / parent_totals[rule.lhs()])
        children = rule.rhs()
        if len(children) == 1 and isinstance(children[0], str):
            lexical_rules[children[0]].append((parent, log_prob))
        elif len(children) == 2:
            left, right = (child.symbol() for child in children)
            binary_rules[left][right].append((parent, log_prob))
        else:
            raise ValueError(f"Rule is outside CNF: {rule}")

    root_counts = Counter(tree.label() for tree in cnf_trees)
    root_scores = {root: log(count / len(cnf_trees)) for root, count in root_counts.items()}
    return lexical_rules, binary_rules, root_scores

### CKY decoding

Each chart cell covers a span. For each label, it stores the best log score and the split that gave that score. Once the chart covers the full sentence, I add the root score and follow the stored splits to rebuild the tree.

In [8]:
def parse_counted(words, grammar):
    lexical_rules, binary_rules, root_scores = grammar
    n = len(words)
    if n == 0:
        return None
    chart = [[{} for _ in range(n + 1)] for _ in range(n)]
    label_order = [[{} for _ in range(n + 1)] for _ in range(n)]

    for start, word in enumerate(words):
        token = word if word in lexical_rules else "<UNK>"
        cell = chart[start][start + 1]
        for label, score in lexical_rules.get(token, ()):
            cell[label] = (score, None)
        label_order[start][start + 1] = {label: i for i, label in enumerate(cell)}

    for width in range(2, n + 1):
        for start in range(n - width + 1):
            end = start + width
            cell = chart[start][end]
            for split in range(start + 1, end):
                left_cell, right_cell = chart[start][split], chart[split][end]
                if not left_cell or not right_cell:
                    continue
                right_order = label_order[split][end]
                for left, (left_score, _) in left_cell.items():
                    rules = binary_rules.get(left)
                    if not rules:
                        continue
                    # Only try child pairs with a rule. Keep chart order for ties.
                    for right in sorted(right_cell.keys() & rules.keys(), key=right_order.get):
                        child_score = left_score + right_cell[right][0]
                        for parent, rule_score in rules[right]:
                            score = child_score + rule_score
                            if parent not in cell or score > cell[parent][0]:
                                cell[parent] = (score, (split, left, right))
            label_order[start][end] = {label: i for i, label in enumerate(cell)}

    root = None
    best_score = float("-inf")
    for label, root_score in root_scores.items():
        if label in chart[0][n]:
            score = chart[0][n][label][0] + root_score
            if score > best_score:
                root, best_score = label, score
    if root is None:
        return None

    def build_tree(start, end, label):
        if end - start == 1:
            return Tree(label, [words[start]])
        split, left, right = chart[start][end][label][1]
        return Tree(label, [build_tree(start, split, left), build_tree(split, end, right)])

    tree = build_tree(0, n, root)
    tree.un_chomsky_normal_form()
    return tree

In [9]:
train_words = {word for tree in train_trees for word in tree.leaves()}


def evaluate_counted(gold_trees, grammar):
    predictions = [parse_counted(tree.leaves(), grammar) for tree in gold_trees]
    lexical_rules, _, _ = grammar
    results = {
        "sentences": len(gold_trees),
        "unseen": sum(any(word not in train_words for word in tree.leaves()) for tree in gold_trees),
        "using_unk": sum(any(word not in lexical_rules for word in tree.leaves()) for tree in gold_trees),
        "complete": sum(tree is not None for tree in predictions),
    }
    for name, labeled in [("Labeled", True), ("Unlabeled", False)]:
        matches = predicted_count = gold_count = 0
        for prediction, gold_tree in zip(predictions, gold_trees):
            gold = brackets(gold_tree, labeled)
            predicted = brackets(prediction, labeled)
            matches += sum((gold & predicted).values())
            predicted_count += sum(predicted.values())
            gold_count += sum(gold.values())
        results[name] = bracket_scores(matches, predicted_count, gold_count)
    return results, predictions

In [10]:
task1_results = {}
task1_records = {}
validation_rows = []
test_rows = []
rare_word_trees = {cutoff: replace_rare_words(train_trees, cutoff) for cutoff in (1, 2)}

for order in (0, 1):
    grammars, validation_results = {}, {}
    for cutoff in (1, 2):
        training_cnf = [to_cnf(tree, order) for tree in rare_word_trees[cutoff]]
        grammars[cutoff] = fit_counted_pcfg(training_cnf)
        validation_results[cutoff], _ = evaluate_counted(validation_trees, grammars[cutoff])
        print(f"Finished validation: order {order}, cutoff {cutoff}")

    best_cutoff = max(validation_results, key=lambda cutoff: validation_results[cutoff]["Labeled"][2])
    test_result, predictions = evaluate_counted(test_trees, grammars[best_cutoff])
    for cutoff, result in validation_results.items():
        validation_rows.append({
            "Order": order, "Cutoff": cutoff, "Selected": cutoff == best_cutoff,
            "Labeled validation F1 (%)": 100 * result["Labeled"][2],
            "Unlabeled validation F1 (%)": 100 * result["Unlabeled"][2],
        })
    test_rows.append({
        "Order": order, "Cutoff": best_cutoff, "Complete parses": test_result["complete"],
        "Labeled test F1 (%)": 100 * test_result["Labeled"][2],
        "Unlabeled test F1 (%)": 100 * test_result["Unlabeled"][2],
    })
    records = [
        score_sentence("test", index, gold, prediction)
        for index, (gold, prediction) in enumerate(zip(test_trees, predictions))
    ]
    task1_records[order] = records
    task1_results[order] = save_evaluation(f"task1_h{order}_test", records)
    save_json(f"task1_h{order}_original_metrics.json", {
        "markov_order": order, "selected_cutoff": best_cutoff,
        "validation_original": validation_results[best_cutoff], "test_original": test_result,
        "aligned_unlabeled_test": task1_results[order],
    })

display(pd.DataFrame(validation_rows).round(2))
display(pd.DataFrame(test_rows).round(2))

Finished validation: order 0, cutoff 1
Finished validation: order 0, cutoff 2
Finished validation: order 1, cutoff 1
Finished validation: order 1, cutoff 2


,Order,Cutoff,Selected,Labeled validation F1 (%),Unlabeled validation F1 (%)
0,0,1,True,56.52,59.70
1,0,2,False,55.70,59.62
2,1,1,True,56.95,60.14
3,1,2,False,56.51,60.70


,Order,Cutoff,Complete parses,Labeled test F1 (%),Unlabeled test F1 (%)
0,0,1,128,53.80,57.26
1,1,1,124,58.41,62.42


The original Task 1 table keeps its punctuation convention. The shared unlabeled scores are saved separately for the combined notebook's cross-task comparison.

In [11]:
run_info.pop("task2_mode", None)
run_info.pop("task3_mode", None)
save_json("run_manifest.json", run_info)
(run_dir / "requirements_runtime.txt").write_text(
    "\n".join(f"{name}=={number}" for name, number in package_versions.items()) + "\n"
)
print(f"Results: {run_dir.resolve()}")

Results: /workspace/scratch/f479fa55acc2/submission_work/checks/task1/execution/results/20261107_125947_118227
